In [1]:
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

In [18]:
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    validation_split=0.2
)
validation_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)
test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    "../Data/Alzheimerr/train_set/",
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    seed=42
)

val_gen = validation_datagen.flow_from_directory(
    "../Data/Alzheimerr/train_set/",
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

test_gen = test_datagen.flow_from_directory(
    "../Data/Alzheimerr/test_set/",
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 5760 images belonging to 4 classes.
Found 1440 images belonging to 4 classes.
Found 800 images belonging to 4 classes.


In [26]:
model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    # Block 1
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Block 2
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Block 3
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(4, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)]
model.summary()

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_35 (Conv2D)              │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_36 (Conv2D)              │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_26 (MaxPooling2D) │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_37 (Conv2D)              │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_38 (Conv2D)              │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_27 (MaxPooling2D) │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_39 (Conv2D)              │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_40 (Conv2D)              │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_28 (MaxPooling2D) │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 4)              │       131,076 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 418,084 (1.59 MB)

 Trainable params: 418,084 (1.59 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
from PIL import Image
import keras.src.utils.image_utils as image_utils
image_utils.pil_image = Image

print("Training the CNN...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=40,
    callbacks=callbacks
)

Training the CNN...
Epoch 1/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 92s 485ms/step - accuracy: 0.3337 - loss: 1.3288 - val_accuracy: 0.5090 - val_loss: 1.1164
Epoch 2/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 85s 470ms/step - accuracy: 0.5259 - loss: 1.0916 - val_accuracy: 0.6285 - val_loss: 0.9405
Epoch 3/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 310s 2s/step - accuracy: 0.6201 - loss: 0.8789 - val_accuracy: 0.6632 - val_loss: 0.8042
Epoch 4/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 175s 967ms/step - accuracy: 0.6590 - loss: 0.7594 - val_accuracy: 0.6917 - val_loss: 0.7293
Epoch 5/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 202s 970ms/step - accuracy: 0.7069 - loss: 0.6862 - val_accuracy: 0.7208 - val_loss: 0.6111
Epoch 6/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 115s 637ms/step - accuracy: 0.7118 - loss: 0.6518 - val_accuracy: 0.7382 - val_loss: 0.6149
Epoch 7/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 66s 366ms/step - accuracy: 0.7406 - loss: 0.6053 - val_accuracy: 0.7715 - val_loss: 0.5359
Epoch 8/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 67s 371ms/step - accur

In [ ]:
import os
# Create the folder if it's missing
if not os.path.exists('../Outputs'):
    os.makedirs('../Outputs')

# This is the line that actually creates the file!
model.save('../Outputs/alzheimer_cnn_model.h5')
print("File successfully created in Outputs folder!")

In [6]:
import os
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model

# 1. LOAD THE MODEL (This fixes the NameError)
# We load the one you saved in the Outputs folder earlier
model_path = '../Outputs/alzheimer_cnn_model.h5'
model = load_model(model_path)
print("Model loaded successfully!")

# 2. Automatically find the first image
folder_path = "../Data/Alzheimerr/test_set/MildDemented/"
all_images = os.listdir(folder_path)
first_image_name = all_images[0] 
img_path = os.path.join(folder_path, first_image_name)
print(f"Testing with image: {img_path}")

# 3. Load and resize the image
img = image.load_img(img_path, target_size=(128, 128))

# 4. Convert to array and prepare for model
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0) 

# 5. Predict!
prediction = model.predict(img_array)
predicted_class = np.argmax(prediction)

# 6. Show results
# Manually defining the classes to keep it simple
classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
print(f"The model predicts this image is: {classes[predicted_class]}")

Model loaded successfully!
Testing with image: ../Data/Alzheimerr/test_set/MildDemented/MildDemented (1).jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
The model predicts this image is: VeryMildDemented


In [ ]:
# Save the CNN model file
model.save('../Outputs/alzheimer_cnn_model.h5')

# Save the CNN accuracy plot
plt.figure()
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('CNN Training Performance')
plt.legend()
plt.savefig('../Outputs/cnn_plot.png')

print("CNN Model and Plot saved to Outputs folder!")